In [1]:
#Loading Libraries
import numpy as np
import torch

import os
from os import listdir
from os.path import isfile, join
from PIL import Image

import torch.optim as optim
from torch.autograd import grad
import time

import torch.utils.data as data_utils

from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize
from torch.utils.data import TensorDataset,DataLoader

from scipy.io import loadmat
from scipy.io import savemat

try:
    from torchvision.transforms import InterpolationMode
    BICUBIC = InterpolationMode.BICUBIC
except ImportError:
    BICUBIC = Image.BICUBIC

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [2]:
nc = 100 #number of classes
d = 2560 #number of features
#Loading data
def load_data(path,file):
    name=path+file
    m=loadmat(name)
    x=torch.tensor(m['feature'])
    return x.float()

path = 'D:/datasets/Cifar100/Clip/' 
trFeatures = torch.empty((0,d))
trY  = torch.empty(0)
for j in range(0,nc):
    x = load_data(path,'train{}.mat'.format(j))
    rep = x.shape[0]
    y =  torch.tensor(j)
    y1 = y.repeat(rep)
    trFeatures = torch.cat((trFeatures,x),dim = 0)
    trY = torch.cat((trY,y1),dim=0)

valFeatures = torch.empty((0,d))
valY  = torch.empty(0)
for j in range(0,nc):
    x = load_data(path,'val{}.mat'.format(j))
    rep = x.shape[0]
    y =  torch.tensor(j)
    y1 = y.repeat(rep)
    valFeatures = torch.cat((valFeatures,x), dim = 0)
    valY = torch.cat((valY,y1),dim=0) 

In [3]:
valY.shape

torch.Size([10000])

In [4]:
from LFA import LFA, PPCA
def fs_ppca(trFeatures, trY, nc, d, q=20):
    """
    Performs Feature Selection using Probabilistic Principal Component Analysis (PPCA)
    for each class and returns a list of feature indices sorted by their relevance.

    Args:
        trFeatures (torch.Tensor): Training features (N x d), where N is the number of samples
                                     and d is the number of features.
        trY (torch.Tensor): Training labels (N).
        nc (int): Number of classes.
        d (int): Number of features.
        q (int, optional): Number of principal components to retain. Defaults to 20.

    Returns:
        torch.Tensor: A tensor of shape (nc x d) containing feature indices sorted in
                      descending order of their relevance for each class.
    """
    index_list = torch.empty((0, d), dtype=torch.int32)
    for i in range(nc):
        data = trFeatures[trY == i, :]
        A_ML=PPCA(data,q)
        
        # Calculate the row sums of squares (feature relevance)
        row_ss = torch.sum(A_ML**2, dim=1)

        # Get the indices sorted by row sums of squares in descending order
        indices = torch.argsort(-row_ss)
        index_list = torch.cat((index_list, indices.unsqueeze(0)), dim=0)

    return index_list

def fs_lfa(trFeatures, trY, nc, d, q=20):
    """
    Performs Feature Selection using Linear Factor Analysis (LFA) for each class.

    Args:
        trFeatures (torch.Tensor): Training features (N x d).
        trY (torch.Tensor): Training labels (N).
        nc (int): Number of classes.
        d (int): Number of features.
        q (int, optional): Number of principal components to retain. Defaults to 20.

    Returns:
        torch.Tensor: A tensor of shape (nc x d) containing feature indices sorted in
                      descending order of their relevance (SNR) for each class.
    """
    index_list = torch.empty((0, d), dtype=torch.int32)
    for i in range(nc):
        data = trFeatures[trY == i, :]
        W_ML, Psi_ML, b  = LFA(data, q, 30)
        snr = torch.sum(W_ML**2, dim=1) / Psi_ML
        indices = torch.argsort(-snr)
        index_list = torch.cat((index_list, indices.unsqueeze(0)), dim=0)
    return index_list

In [5]:
# ELF
def ELF_st(X, r, d):
    """
    Initializes parameters for Exploratory Latent Factor (ELF) model.

    Args:
        X (torch.Tensor): Data matrix (n x d).
        r (int): Number of latent factors.
        d (int): Number of features.

    Returns:
        tuple: Initial latent factors (Gamma) and noise variance (Psi).
    """
    u, s, v = torch.linalg.svd(X)
    Gamma = u[:, :r] * s[:r]
    Psi = torch.ones(d, dtype=torch.float32)
    return (Gamma, Psi)

def ELF(X0, Gamma, Psi, n, epochs, r, d, tolerance=0.1):
    """
    Performs Exploratory Latent Factor (ELF) algorithm.

    Args:
        X0 (torch.Tensor): Centered data matrix (n x d).
        Gamma (torch.Tensor): Initial latent factors (n x r).
        Psi (torch.Tensor): Initial noise variance (d).
        n (int): Number of samples.
        epochs (int): Maximum number of iterations.
        r (int): Number of latent factors.
        d (int): Number of features.
        tolerance (float, optional): Convergence tolerance. Defaults to 0.1.

    Returns:
        torch.Tensor: Signal-to-noise ratio (SNR) for each feature (d).
    """
    W = torch.ones((d, r))
    for ep in range(epochs):
        # print(ep)
        # Updating W
        W_new = X0.t() @ Gamma
        M1 = torch.linalg.inv(Gamma.t() @ Gamma)
        W_new = W_new @ M1
        # Updating Gamma
        m1 = torch.diag(1 / Psi) @ W_new
        invMat = torch.linalg.inv(W_new.t() @ m1)
        Gamma = X0 @ m1 @ invMat
        # Orthogonalisation
        u, s, v = torch.linalg.svd(Gamma)
        M = v.t() @ torch.diag(s)
        W_new = (W_new @ M) / np.sqrt(n)
        Gamma = u[:, :r] * np.sqrt(n)
        # Updating weights (Psi)
        Xhat = Gamma @ W_new.t()
        Psi = torch.mean((X0 - Xhat)**2, dim=0)
        Psi[Psi < 0.01] = 0.01

        if torch.sqrt(torch.sum((W - W_new)**2)) < tolerance:
            print('ep', ep)
            break
        W = W_new
    snr = torch.sum(W_new**2, dim=1) / Psi
    return snr



In [6]:
# HPCA
T_hpca = 5
r_hpca = 10
def heteroPCA(Cov, r, T, d):
    """
    Performs Heteroscedastic Principal Component Analysis (HPCA).

    Args:
        Cov (torch.Tensor): Covariance matrix (d x d).
        r (int): Number of principal components.
        T (int): Number of iterations.
        d (int): Number of features.

    Returns:
        tuple: Principal components (u) and the modified covariance matrix (N_tilda).
    """
    d_indi = torch.arange(d)
    N_prev = Cov - torch.diag(torch.diagonal(Cov))
    for t in range(T):
        # print(t)
        u, s, v = torch.linalg.svd(N_prev)
        N_tilda = u[:, :r] @ torch.diag(s[:r]) @ v[:r, :]
        N_tilda_D = torch.diagonal(N_tilda)
        N_prev[d_indi, d_indi] = N_tilda_D
    return (u[:, :r], N_tilda)



In [8]:
trY.shape

torch.Size([50000])

In [9]:
index_dic={}
time_dic={}

In [10]:
# Feature Selection using PPCA
start = time.time()
index_lst = fs_ppca(trFeatures, trY, nc, d, q=20)
index_dic['PPCA'] = index_lst
end = time.time()
time_dic['PPCA'] = end-start

In [11]:
time_dic

{'PPCA': 14.173759460449219}

In [12]:
# Feature Selection using LFA
index_lst = fs_lfa(trFeatures, trY, nc, d, q=20)
index_dic['LFA'] = index_lst

In [17]:
# Feature Selection using ELF
def fs_ELF(trFeatures, trY, nc, d, r=r_elf, epochs=epochs_elf):
    """
    Performs Estimation by Latent Factor (ELF) based feature selection for each class.

    Args:
        trFeatures (torch.Tensor): Training features (N x d).
        trY (torch.Tensor): Training labels (N).
        nc (int): Number of classes.
        d (int): Number of features.
        r (int, optional): Number of latent factors. Defaults to r_elf (10).
        epochs (int, optional): Maximum number of ELF iterations. Defaults to epochs_elf (100).
        device (str, optional): Device to run computations on ('cpu' or 'cuda'). Defaults to 'cpu'.

    Returns:
        torch.Tensor: A tensor of shape (nc x d) containing feature indices sorted in
                      descending order of their relevance (SNR) for each class.
    """
    idx_list = torch.empty((0, d), dtype=torch.int32).to(device)

    for i in range(nc):
        X = trFeatures[trY == i, :]
        mu = torch.mean(X, dim=0)
        X0 = (X - mu)

        Gamma, Psi = ELF_st(X0, r, d)
        n = X0.shape[0]
        snr = ELF(X0, Gamma, Psi, n, epochs, r, d)
        idx = torch.argsort(-snr).to(device)
        idx_list = torch.cat((idx_list, idx.unsqueeze(0)), dim=0)
    return idx_list
    
epochs_elf = 10
r_elf = 10
index_lst = fs_ELF(trFeatures, trY, nc, d, r=r_elf, epochs=epochs_elf)
index_dic['ELF'] = index_lst

In [19]:
# Feature Selection using HPCA
def fs_HPCA(trFeatures, trY, nc, d, r=r_hpca, T=T_hpca):
    """
    Performs Feature Selection using Heteroscedastic Probabilistic Principal Component Analysis (HPCA)
    for each class.

    Args:
        trFeatures (torch.Tensor): Training features (N x d).
        trY (torch.Tensor): Training labels (N).
        nc (int): Number of classes.
        d (int): Number of features.
        r (int, optional): Number of principal components for HPCA. Defaults to r_hpca (10).
        T (int, optional): Number of iterations for HPCA. Defaults to T_hpca (5).
        device (str, optional): Device to run computations on ('cpu' or 'cuda'). Defaults to 'cpu'.

    Returns:
        torch.Tensor: A tensor of shape (nc x d) containing feature indices sorted in
                      descending order of their relevance (SNR-like metric) for each class.
    """
    idx_list = torch.empty((0, d), dtype=torch.int32).to(device)
    for i in range(nc):
        X = trFeatures[trY == i, :]
        mu = torch.mean(X, dim=0)
        # std = torch.std(X,dim=0) # Standard deviation is not used here
        X_centered_t = (X - mu).t()
        Cov = torch.cov(X_centered_t) + 0.01 * torch.eye(d)
        u, _ = heteroPCA(Cov, r, T, d)

        Gamma = X_centered_t.t() @ u
        W = u

        n = X_centered_t.shape[1]
        u_gamma, s_gamma, v_gamma = torch.linalg.svd(Gamma)
        M = v_gamma.t() @ torch.diag(s_gamma)
        W_new = (W @ M) / np.sqrt(n)
        Gamma_new = u_gamma[:, :r] * np.sqrt(n)

        X_hat = Gamma_new @ W_new.t()

        Psi = torch.mean((X_centered_t.t() - X_hat)**2, dim=0)
        Sig = torch.sum(W_new**2, dim=1)

        snr_e = Sig / Psi
        idx = torch.argsort(-snr_e).to(device)
        idx_list = torch.cat((idx_list, idx.unsqueeze(0)), dim=0)
    return idx_list

index_lst = fs_HPCA(trFeatures, trY, nc, d, r=r_hpca, T=T_hpca)
index_dic['HPCA'] = index_lst

In [15]:
mu = torch.mean(trFeatures,dim = 0)
std = torch.std(trFeatures,dim=0)

trfeatures = (trFeatures-mu)/std
valfeatures = (valFeatures -mu)/std

In [22]:
class MahalanobisClassifier:
    """
    A classifier that uses Mahalanobis distance for classification.
    """
    def __init__(self, device='cpu'):
        """
        Initializes the MahalanobisClassifier.

        Args:
            device (str, optional): The device to perform computations on ('cpu' or 'cuda'). Defaults to 'cpu'.
        """
        self.device = device
        self.means = []
        self.inv_covariances = []

    def _calculate_class_parameters(self, index_list, features_train, trY, nc, nf):
        """
        Calculates the mean and inverse covariance matrix for the selected features of each class.

        Args:
            index_list (torch.Tensor): Tensor of feature indices for each class (nc x nf).
            features_train (torch.Tensor): Training features (N x d).
            trY (torch.Tensor): Training labels (N).
            nc (int): Number of classes.
            nf (int): Number of selected features.
        """
        self.means = []
        self.inv_covariances = []
        for i in range(nc):
            indices = index_list[i]
            features = features_train[trY == i, :]
            features_selected = features[:, indices]

            mean = torch.mean(features_selected, dim=0).to(self.device)
            self.means.append(mean)

            cov = torch.cov(features_selected.t()) + 0.1 * torch.eye(nf, device=self.device)
            inv_cov = torch.linalg.inv(cov)
            self.inv_covariances.append(inv_cov)

    def _mahalanobis_distance(self, x, inv_cov):
        """
        Calculates the Mahalanobis distance.

        Args:
            x (torch.Tensor): Data point (1 x nf).
            inv_cov (torch.Tensor): Inverse covariance matrix (nf x nf).

        Returns:
            torch.Tensor: Mahalanobis distance.
        """
        diff = (x - self.means[self.current_class]).to(self.device)
        dist = (diff @ inv_cov) @ diff.t()
        return dist

    def classify(self, features_valid, index_list, nc, n_val):
        """
        Classifies validation features based on Mahalanobis distance to class means.

        Args:
            features_valid (torch.Tensor): Validation features (N_val x d).
            index_list (torch.Tensor): Tensor of feature indices for each class (nc x nf).
            nc (int): Number of classes.
            n_val (int): Number of validation samples.

        Returns:
            torch.Tensor: Predicted class labels for the validation set.
        """
        out = torch.zeros(n_val, nc, device=self.device)
        for i in range(nc):
            self.current_class = i
            indices = index_list[i]
            x_valid_selected = features_valid[:, indices].to(self.device)
            inv_cov = self.inv_covariances[i]
            for j in range(n_val):
                out[j, i] = self._mahalanobis_distance(x_valid_selected[j], inv_cov)
        predicted_labels = torch.argmin(out, dim=1).cpu()
        return predicted_labels

def calculate_accuracy(classifier, features_train, features_valid, index_list, valY, trY, nc, nf):
    """
    Calculates the classification accuracy using the MahalanobisClassifier.

    Args:
        classifier (MahalanobisClassifier): An instance of the MahalanobisClassifier.
        features_train (torch.Tensor): Training features (N x d).
        features_valid (torch.Tensor): Validation features (N_val x d).
        index_list (torch.Tensor): Tensor of feature indices for each class (nc x nf).
        valY (torch.Tensor): Validation labels (N_val).
        trY (torch.Tensor): Training labels (N).
        nc (int): Number of classes.
        nf (int): Number of selected features.

    Returns:
        float: The classification accuracy.
    """
    n_val = features_valid.shape[0]
    classifier._calculate_class_parameters(index_list, features_train, trY, nc, nf)
    predictions = classifier.classify(features_valid, index_list, nc, n_val)
    accuracy = np.round((torch.sum(predictions == valY.cpu()).item() / n_val), 4)
    return accuracy

nf_list = [750, 1000, 1250, 1500, 1750, 2000, 2250, 2560]
accuracy_results = {}
classifier = MahalanobisClassifier(device)
for method, indices in index_dic.items():
    accuracy_list = []
    for nf in nf_list:
        print(f"Evaluating {method} with {nf} features...")
        index_sel = indices[:, :nf]
        accuracy = calculate_accuracy(classifier, trfeatures.to(device), valfeatures.to(device), index_sel, valY, trY, nc, nf)
        accuracy_list.append(accuracy)
        print(f"{method} ({nf} features) Accuracy: {accuracy}")
    accuracy_results[method] = accuracy_list

print("\nClassification Accuracy for Different Number of Features:")
for method, accuracies in accuracy_results.items():
    print(f"{method}:")
    for i, nf in enumerate(nf_list):
        print(f"  {nf} features: {accuracies[i]}")

Evaluating PPCA with 750 features...
PPCA (750 features) Accuracy: 0.6544
Evaluating PPCA with 1000 features...
PPCA (1000 features) Accuracy: 0.693
Evaluating PPCA with 1250 features...
PPCA (1250 features) Accuracy: 0.7085
Evaluating PPCA with 1500 features...
PPCA (1500 features) Accuracy: 0.7214
Evaluating PPCA with 1750 features...
PPCA (1750 features) Accuracy: 0.7251
Evaluating PPCA with 2000 features...
PPCA (2000 features) Accuracy: 0.7301
Evaluating PPCA with 2250 features...
PPCA (2250 features) Accuracy: 0.7287
Evaluating PPCA with 2560 features...
PPCA (2560 features) Accuracy: 0.7283
Evaluating LFA with 750 features...
LFA (750 features) Accuracy: 0.6444
Evaluating LFA with 1000 features...
LFA (1000 features) Accuracy: 0.6709
Evaluating LFA with 1250 features...
LFA (1250 features) Accuracy: 0.6926
Evaluating LFA with 1500 features...
LFA (1500 features) Accuracy: 0.702
Evaluating LFA with 1750 features...
LFA (1750 features) Accuracy: 0.7114
Evaluating LFA with 2000 fea